In [ ]:
import sys
sys.path.insert(0, "..")
import matplotlib.pyplot as plt
from src.data.loaders import load_iam
from src.data.preprocessing import preprocess
from src.ocr.baseline import extract_text
from src.ocr.evaluate import compute_metrics

In [ ]:
_, iam_val = load_iam(max_train=10, max_val=100)
print(f"Test samples: {len(iam_val)}")

In [ ]:
results = [extract_text(preprocess(doc.image)) for doc in iam_val]
references = [doc.text for doc in iam_val]
metrics = compute_metrics([r.text for r in results], references)
print(f"Tesseract CER: {metrics['cer']:.4f}")
print(f"Tesseract WER: {metrics['wer']:.4f}")

In [ ]:
failures = [(doc, r) for doc, r in zip(iam_val, results) if r.text.lower() != doc.text.lower()]
print(f"Failure cases: {len(failures)} / {len(iam_val)}")
fig, axes = plt.subplots(min(3, len(failures)), 2, figsize=(14, 10))
if len(failures) > 0:
    axes = axes.reshape(-1, 2) if len(failures) > 1 else [axes]
    for i, (doc, r) in enumerate(failures[:3]):
        axes[i][0].imshow(doc.image, cmap="gray")
        axes[i][0].set_title("Image")
        axes[i][0].axis("off")
        axes[i][1].text(0.05, 0.5, f"REF: {doc.text}\nOCR: {r.text}",
                        transform=axes[i][1].transAxes, fontsize=9, va="center", wrap=True)
        axes[i][1].axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
confs = [r.confidence for r in results]
plt.figure(figsize=(8, 4))
plt.hist(confs, bins=20, color="#FF6B6B", edgecolor="white")
plt.title("Tesseract Confidence Distribution")
plt.xlabel("Confidence")
plt.show()